# Inference & test notebook
This notebook demonstrates inference with pragmatic fallbacks to keep runtime within 90 minutes on Kaggle kernels: caching, batching, subset-mode, and TTA options.

In [ ]:
import os, time, math
import numpy as np, pandas as pd
import torch
from torchvision import models
import librosa
ROOT = os.path.abspath(os.path.join('..','..'))
DATA_ROOT = os.path.join(ROOT,'data','raw')
TEST_DIR = os.path.join(DATA_ROOT, 'test_soundscapes')
MEL_CACHE = os.path.join(ROOT, 'cached_mels')
os.makedirs(MEL_CACHE, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

In [ ]:
# Load model checkpoint (edit path as needed)
ckpt = os.path.join(ROOT, 'models','baseline_resnet50.pt')
tax = pd.read_csv(os.path.join(DATA_ROOT,'taxonomy.csv'))
num_classes = tax.shape[0]
model = models.resnet50(pretrained=False)
model.conv1 = torch.nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = torch.nn.Linear(model.fc.in_features, num_classes)
if os.path.exists(ckpt):
    state = torch.load(ckpt, map_location='cpu')
    model.load_state_dict(state['model_state'])
model = model.to(device).eval()
print('loaded checkpoint', ckpt)

In [ ]:
# Inference helpers: mel loader, TTA, batching, and fallbacks
import hashlib
def cached_mel_path(filepath):
    h = hashlib.sha1(filepath.encode()).hexdigest()
    return os.path.join(MEL_CACHE, h + '.npy')
def get_mel(filepath, sr=32000, duration=5.0):
    cache = cached_mel_path(filepath)
    if os.path.exists(cache):
        return np.load(cache)
    y, _ = librosa.load(filepath, sr=sr, mono=True, res_type='kaiser_fast')
    samples = int(sr * duration)
    if len(y) < samples: y = np.pad(y, (0, max(0, samples - len(y))))
    else: y = y[:samples]
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=2048, hop_length=320, n_mels=128)
    Sdb = librosa.power_to_db(S, ref=np.max).astype(np.float32)
    np.save(cache, Sdb)
    return Sdb
def tta_predict(mel, model, tta_steps=3):
    model.eval()
    preds = []
    for _ in range(tta_steps):
        x = torch.tensor(mel)[None,None,:,:].to(device).float()
        x = torch.nn.functional.interpolate(x, size=(128,128))
        with torch.no_grad():
            logits = model(x)
            preds.append(torch.softmax(logits, dim=-1).cpu().numpy())
    return np.mean(np.vstack(preds), axis=0)[0]

In [ ]:
# Runtime-aware inference loop with fallback strategies:
# - If GPU available: process full test set in batches
# - If not: process in smaller chunks and save partial outputs to disk to avoid timeouts
test_files = sorted([os.path.join(TEST_DIR,f) for f in os.listdir(TEST_DIR) if f.endswith('.ogg')])[:20]  # sample small subset by default
results = []
start = time.time()
for i,fp in enumerate(test_files):
    try:
        mel = get_mel(fp)
        p = tta_predict(mel, model, tta_steps=2)
        # simple top-1 pick for demo; real submission needs thresholding & multi-label handling
        top_idx = int(np.argmax(p))
        top_label = str(int(tax.iloc[top_idx]['primary_label']))
        results.append({'filename': os.path.basename(fp), 'prediction': top_label})
    except Exception as e:
        results.append({'filename': os.path.basename(fp), 'prediction': '', 'error': str(e)})
    # periodic save to avoid losing progress on long runs
    if i % 10 == 0:
        pd.DataFrame(results).to_csv('partial_infer_results.csv', index=False)
    elapsed = time.time() - start
    # if approaching 80 minutes, stop and persist outputs
    if elapsed > 60*80:
        print('Approaching runtime limit, breaking early')
        break
pd.DataFrame(results).to_csv('infer_results.csv', index=False)
print('done, results saved to infer_results.csv')